# 03 — Explorer les résultats (Pandas & visualisations simples)

**Prérequis :** avoir des fichiers dans `data/processed/` (lance `python scripts/sync_wikidata.py --full` si besoin).

Ce notebook propose des **pistes d'exploration** — la liste n'est pas exhaustive : c'est un point de départ pour ton propre travail.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

# Chemins depuis le dossier notebooks/
ROOT = Path("..")
DATA = ROOT / "data" / "processed"
CSV_PATH = DATA / "jumelages.csv"
GRAPH_PATH = DATA / "graphe_jumelages.json"
STATS_PATH = DATA / "stats.json"
META_PATH = DATA / "metadata.json"

## 1. Charger le CSV (source de vérité)

Chaque ligne = **une paire** de villes jumelées (sans doublon A↔B).

In [ ]:
df = pd.read_csv(CSV_PATH, dtype={"ville_A_id": str, "ville_B_id": str})

print(f"Nombre de jumelages : {len(df)}")
print(f"Colonnes : {list(df.columns)}")
df.head()

## 2. Aperçu rapide (`info`, `describe`)

Utile pour repérer les valeurs manquantes (coords, pays…).

In [ ]:
df.info()
print()
df[["ville_A_lon", "ville_A_lat", "ville_B_lon", "ville_B_lat"]].describe()

## 3. Filtrer — exemples de questions

Tu peux adapter ces filtres à ta curiosité.

In [ ]:
# Jumelages impliquant la France (ville A ou B)
france = df[(df["ville_A_pays"] == "FR") | (df["ville_B_pays"] == "FR")]
print(f"Liens avec la France : {len(france)}")
france.head(10)

In [ ]:
# Jumelages *internationaux* uniquement (pays A ≠ pays B)
international = df[df["ville_A_pays"] != df["ville_B_pays"]].dropna(subset=["ville_A_pays", "ville_B_pays"])
print(f"Jumelages internationaux : {len(international)}")
international.head(10)

In [ ]:
# Chercher une ville par nom (insensible à la casse)
nom = "Paris"  # ← change ce nom

mask = df["ville_A_nom"].str.contains(nom, case=False, na=False) | df["ville_B_nom"].str.contains(nom, case=False, na=False)
resultat = df[mask]
print(f"Jumelages contenant « {nom} » : {len(resultat)}")
resultat

## 4. Villes les plus jumelées (degré du graphe)

On compte combien de fois chaque ville apparaît dans le CSV.

In [ ]:
# Empiler les noms des deux côtés de chaque lien
villes_a = df[["ville_A_id", "ville_A_nom"]].rename(columns={"ville_A_id": "id", "ville_A_nom": "nom"})
villes_b = df[["ville_B_id", "ville_B_nom"]].rename(columns={"ville_B_id": "id", "ville_B_nom": "nom"})
toutes = pd.concat([villes_a, villes_b])

top = toutes.groupby("nom").size().sort_values(ascending=False).head(15)
print("Top 15 villes les plus connectées :")
top

In [ ]:
# Bar chart simple
fig, ax = plt.subplots(figsize=(10, 5))
top.sort_values().plot(kind="barh", ax=ax, color="#58a6ff")
ax.set_title("Top villes jumelées (nombre de partenaires)")
ax.set_xlabel("Nombre de jumelages")
plt.tight_layout()
plt.show()

## 5. Répartition par pays

In [ ]:
pays_a = df["ville_A_pays"].dropna()
pays_b = df["ville_B_pays"].dropna()
comptage_pays = pd.concat([pays_a, pays_b]).value_counts().head(15)

print(comptage_pays)

fig, ax = plt.subplots(figsize=(8, 5))
comptage_pays.plot(kind="bar", ax=ax, color="#f0883e")
ax.set_title("Pays les plus présents dans les jumelages (top 15)")
ax.set_ylabel("Occurrences")
plt.xticks(rotation=45, ha="right")
plt.tight_layout()
plt.show()

## 6. Distance approximative entre jumelles (formule haversine)

Utile pour voir si les jumelages sont plutôt **locaux** ou **intercontinentaux**.

In [ ]:
import math


def distance_km(lon1, lat1, lon2, lat2):
    """Distance à vol d'oiseau entre deux points GPS (en km)."""
    r = 6371.0
    p1, p2 = math.radians(lat1), math.radians(lat2)
    dlat = math.radians(lat2 - lat1)
    dlon = math.radians(lon2 - lon1)
    a = math.sin(dlat / 2) ** 2 + math.cos(p1) * math.cos(p2) * math.sin(dlon / 2) ** 2
    return 2 * r * math.asin(math.sqrt(a))


coords_ok = df.dropna(subset=["ville_A_lon", "ville_A_lat", "ville_B_lon", "ville_B_lat"]).copy()

coords_ok["distance_km"] = coords_ok.apply(
    lambda row: distance_km(row.ville_A_lon, row.ville_A_lat, row.ville_B_lon, row.ville_B_lat),
    axis=1,
)

print(f"Distance moyenne : {coords_ok['distance_km'].mean():.0f} km")
print(f"Distance max     : {coords_ok['distance_km'].max():.0f} km")
coords_ok.nlargest(5, "distance_km")[["ville_A_nom", "ville_B_nom", "distance_km"]]

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
coords_ok["distance_km"].plot(kind="hist", bins=30, ax=ax, color="#58a6ff", edgecolor="white")
ax.set_title("Distribution des distances entre villes jumelées")
ax.set_xlabel("Distance (km)")
plt.tight_layout()
plt.show()

## 7. Explorer le JSON graphe

Format pensé pour la future carte interactive : `nodes` (points) + `links` (arêtes).

In [ ]:
with GRAPH_PATH.open(encoding="utf-8") as f:
    graphe = json.load(f)

print(f"Nodes : {len(graphe['nodes'])}")
print(f"Links : {len(graphe['links'])}")
print()
print("Exemple node :", graphe["nodes"][0])
print("Exemple link :", graphe["links"][0])

## 8. Scatter plot — villes sur une mini-carte

Projection très simplifiée (lon/lat en axes X/Y), suffisante pour une intuition.

In [ ]:
nodes = pd.DataFrame(graphe["nodes"]).dropna(subset=["lon", "lat"])

fig, ax = plt.subplots(figsize=(10, 5))
ax.scatter(nodes["lon"], nodes["lat"], s=40, alpha=0.7, c="#58a6ff")

for _, row in nodes.iterrows():
    ax.annotate(row["name"], (row["lon"], row["lat"]), fontsize=8, alpha=0.8)

ax.set_title("Villes jumelées (aperçu)")
ax.set_xlabel("Longitude")
ax.set_ylabel("Latitude")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 9. Lire les agrégats pré-calculés (`stats.json`)

In [ ]:
with STATS_PATH.open(encoding="utf-8") as f:
    stats = json.load(f)

print("KPIs :")
for cle, valeur in stats["kpis"].items():
    print(f"  {cle:25} {valeur}")

print()
print(f"Distance moyenne des arcs : {stats.get('avg_arc_km')} km")
print()
print("Top villes (déjà calculé pour le site) :")
pd.DataFrame(stats["top_cities"])

## 10. Matrice pays ↔ pays (pour chord diagram)

La matrice indique combien de jumelages relient deux pays.

In [ ]:
chord = stats["country_chord"]
labels = chord["labels"]
matrix = pd.DataFrame(chord["matrix"], index=labels, columns=labels)

print("Matrice pays → pays (extraits non nuls) :")
matrix.where(matrix > 0).stack().sort_values(ascending=False).head(10)

## 11. Métadonnées — fraîcheur des données

In [ ]:
with META_PATH.open(encoding="utf-8") as f:
    meta = json.load(f)

pd.Series(meta)

## 12. Idées d'exploration libre (non exhaustif)

Coche mentalement ce que tu pourrais ajouter :

- [ ] Exporter un sous-ensemble en Excel : `df.to_excel("mes_jumelages.xlsx")` (pip install openpyxl)
- [ ] Compter les jumelages **intra-pays** vs **internationaux**
- [ ] Carte choroplèthe avec **Plotly** ou **Folium** (pip install plotly / folium)
- [ ] Graphe réseau avec **NetworkX** : `pip install networkx`
- [ ] Comparer deux exports CSV datés (diff entre deux runs)
- [ ] Croiser avec une table continents (`countries.json`)
- [ ] Trouver les villes **sans coordonnées** (non cartographiables)
- [ ] Analyser l'évolution après un `--full` vs delta
- [ ] Importer dans **Power BI** ou **Tableau** pour un dashboard perso

---

**Rappel :** les données viennent de Wikidata (P190) — cite toujours la source et la date (`metadata.json`) dans un rapport ou une présentation.

In [ ]:
# Bonus : villes sans coordonnées (limitent la carte)
sans_coords = df[
    df[["ville_A_lon", "ville_A_lat"]].isna().any(axis=1)
    | df[["ville_B_lon", "ville_B_lat"]].isna().any(axis=1)
]
print(f"Liens avec au moins une ville sans GPS : {len(sans_coords)}")
sans_coords.head()